# Module 00 — Image Processing Fundamentals (SOLUTIONS)

This notebook contains the complete solutions for all three exercises.
Try the main `notebook.ipynb` first!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import urllib.request, io

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

def fetch_image(url, size=(256, 256)):
    with urllib.request.urlopen(url) as resp:
        data = resp.read()
    img = Image.open(io.BytesIO(data)).convert('RGB').resize(size)
    return np.array(img)

URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
try:
    img_rgb = fetch_image(URL)
except Exception:
    img_rgb = np.zeros((256, 256, 3), dtype=np.uint8)
    img_rgb[:128, :128] = [220, 60, 60]
    img_rgb[:128, 128:] = [60, 180, 60]
    img_rgb[128:, :128] = [60, 60, 220]
    img_rgb[128:, 128:] = [180, 180, 60]

img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
clahe_cv2 = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(img_gray)

## Solution — Exercise 1: Unsharp Masking

In [ ]:
def unsharp_mask(image: np.ndarray, sigma: float = 1.0, amount: float = 1.0) -> np.ndarray:
    """Sharpen image using unsharp masking."""
    ksize = int(6 * sigma + 1) | 1  # ensure odd
    blurred = cv2.GaussianBlur(image.astype(np.float32), (ksize, ksize), sigma)
    # high-frequency component
    detail = image.astype(np.float32) - blurred
    sharpened = image.astype(np.float32) + amount * detail
    return sharpened.clip(0, 255).astype(np.uint8)


sharpened = unsharp_mask(img_gray, sigma=2, amount=1.5)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_gray, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(sharpened, cmap='gray'); axes[1].set_title('Unsharp mask (σ=2, a=1.5)'); axes[1].axis('off')
diff = sharpened.astype(int) - img_gray.astype(int)
axes[2].imshow(diff + 128, cmap='bwr', vmin=0, vmax=255); axes[2].set_title('Enhancement map'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Solution — Exercise 2: Background Subtraction

In [ ]:
rng = np.random.default_rng(42)
frames = []
for t in range(10):
    frame = (rng.random((128, 128)) * 60 + 100).clip(0, 255).astype(np.uint8)
    x = 20 + t * 8
    frame[40:70, x:x+30] = 220
    frames.append(frame)

def subtract_background(frames, threshold=30):
    """Mean-background subtraction."""
    stack = np.stack(frames, axis=0).astype(np.float32)
    background = stack.mean(axis=0)
    masks = []
    for frame in frames:
        diff = np.abs(frame.astype(np.float32) - background)
        masks.append(diff > threshold)
    return masks

masks = subtract_background(frames, threshold=25)

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
for t, (ax_f, ax_m) in enumerate(zip(axes[0], axes[1])):
    ax_f.imshow(frames[t], cmap='gray', vmin=0, vmax=255)
    ax_f.set_title(f't={t}'); ax_f.axis('off')
    ax_m.imshow(masks[t], cmap='gray'); ax_m.axis('off')
axes[0, 0].set_ylabel('Frames', fontsize=10)
axes[1, 0].set_ylabel('Foreground masks', fontsize=10)
plt.suptitle('Background Subtraction', fontsize=12)
plt.tight_layout(); plt.show()

## Solution — Exercise 3: CLAHE from First Principles

In [ ]:
def clahe_scratch(image: np.ndarray, clip_limit: float = 2.0, grid: int = 8) -> np.ndarray:
    """Simplified tile-wise CLAHE without bilinear interpolation."""
    H, W = image.shape
    out = np.zeros_like(image)
    tile_h = H // grid
    tile_w = W // grid

    for r in range(grid):
        for c in range(grid):
            y0, y1 = r * tile_h, (r + 1) * tile_h
            x0, x1 = c * tile_w, (c + 1) * tile_w
            # Handle edge tiles
            if r == grid - 1: y1 = H
            if c == grid - 1: x1 = W
            tile = image[y0:y1, x0:x1]
            n_pixels = tile.size

            # Compute histogram
            hist, _ = np.histogram(tile.flatten(), bins=256, range=(0, 256))

            # Clip and redistribute
            clip_val = max(1, int(clip_limit * n_pixels / 256))
            excess = np.sum(np.maximum(0, hist - clip_val))
            hist = np.minimum(hist, clip_val)
            hist = hist + excess // 256  # redistribute uniformly

            # Compute CDF-based LUT
            cdf = hist.cumsum().astype(np.float64)
            cdf_min = cdf[cdf > 0].min() if (cdf > 0).any() else 0
            lut = np.clip(
                (cdf - cdf_min) / (n_pixels - cdf_min + 1e-6) * 255, 0, 255
            ).astype(np.uint8)

            # Apply LUT
            out[y0:y1, x0:x1] = lut[tile]

    return out


result = clahe_scratch(img_gray, clip_limit=2.0, grid=8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_gray, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(result, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('CLAHE (scratch)'); axes[1].axis('off')
axes[2].imshow(clahe_cv2, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('CLAHE (OpenCV)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

print(f'Mean absolute difference vs OpenCV CLAHE: {np.abs(result.astype(int) - clahe_cv2.astype(int)).mean():.2f}')